# Lecture 9: Hardware and Software

*Notebook for UMich EECS 498 (Justin Johnson, Fall 2019), Lecture 9.*

Deep learning took off when GPUs became fast and software made them easy to use. This lecture is about both.

**What you'll learn**
1. CPUs vs. GPUs vs. TPUs, and why deep learning loves GPUs.
2. Why matrix multiplication is fast, and number formats like FP16.
3. What a deep learning framework does: we'll **build a tiny PyTorch** ourselves.
4. Static vs. dynamic computational graphs.
5. PyTorch, TensorFlow, and JAX.
6. How training is split across many GPUs.

In [ ]:
import time
import numpy as np
from sklearn.datasets import load_digits

## 1. CPU vs. GPU vs. TPU

* **CPU:** a few powerful cores (typically 4–64). Great at **sequential** tasks and complicated logic.
* **GPU:** thousands of simpler cores that all do the same operation on different data. Great at **parallel** tasks. Originally built for graphics (shading millions of pixels the same way); **CUDA** (2007) made them programmable for any computation.
* **TPU:** Google's chip designed specifically for neural networks: big units that do matrix multiplication.

Deep learning is mostly **matrix multiplication** and convolution, which are exactly the kind of massively parallel work GPUs are built for. Some GPUs' peak speeds (in trillions of operations per second, TFLOP/s):

| GPU | Year | FP32 TFLOP/s | 16-bit "tensor core" TFLOP/s | Memory |
|---|---|---|---|---|
| GTX 580 (AlexNet used 2) | 2010 | 1.6 | — | 3 GB |
| V100 | 2017 | 15.7 | 125 | 32 GB |
| A100 | 2020 | 19.5 | 312 | 80 GB |
| H100 | 2022 | 67 | 989 | 80 GB |

Most of the speedup comes from **tensor cores**, which multiply small matrices in 16-bit precision in one step.

## 2. Why matrix multiplication is fast

Every entry of $C = AB$ is an independent dot product (a row of $A$ times a column of $B$), so thousands of cores can compute different entries at the same time. Even on a CPU, a well-written library (NumPy calls one called BLAS) is enormously faster than computing the dot products one at a time in Python:

In [ ]:
N = 200
rng = np.random.default_rng(0)
A = rng.random((N, N))
B = rng.random((N, N))

start = time.time()
C_loops = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        C_loops[i, j] = np.dot(A[i], B[:, j])        # one dot product at a time
time_loops = time.time() - start

start = time.time()
C_fast = A @ B                                         # the whole matrix multiply at once
time_fast = time.time() - start

print(f"one dot product at a time: {time_loops:.3f} s")
print(f"A @ B:                     {time_fast * 1000:.2f} ms  ({time_loops / time_fast:.0f}x faster)")
print("same answer:", np.allclose(C_loops, C_fast))

**Computation vs. memory.** A chip has two speed limits: how fast it can **compute** and how fast it can **read memory**. A large matrix multiply does a lot of math for each number it reads ($2N^3$ operations on $3N^2$ numbers), so it keeps the chip busy. Adding two matrices does just **one** operation per number read, so the chip mostly waits for memory:

In [ ]:
N = 1024
A = rng.random((N, N), dtype=np.float32)
B = rng.random((N, N), dtype=np.float32)

start = time.time(); A @ B; time_matmul = time.time() - start
start = time.time(); A + B; time_add = time.time() - start

print(f"matrix multiply: {2 * N**3 / time_matmul / 1e9:6.1f} billion operations per second")
print(f"matrix add:      {N**2 / time_add / 1e9:6.1f} billion operations per second")

Same hardware, very different speeds. That's why neural networks are built mostly out of matrix multiplies.

### Number formats: 32-bit vs. 16-bit
Using **16-bit** numbers instead of 32-bit halves the memory and unlocks tensor cores. The catch: 16-bit floats (**FP16**) have a small range. Big numbers overflow and tiny ones become zero:

In [ ]:
for value in [3.14159, 70000.0, 1e-8]:
    with np.errstate(over="ignore"):
        print(f"{value:>10g} in FP32: {np.float32(value):<12g} in FP16: {float(np.float16(value)):g}")

That's a problem for training, because many gradients are tiny. **Mixed-precision training** uses **loss scaling**: multiply the loss by a big number (say 1024) before backprop, which scales every gradient up into FP16's range (by the chain rule); then divide by 1024 before updating the weights.

In [ ]:
tiny_gradient = 2e-8
print("stored in FP16:                      ", float(np.float16(tiny_gradient)), " <- lost!")
print("scaled by 1024, stored, then unscaled:", float(np.float16(tiny_gradient * 1024)) / 1024, " <- survives")

**BF16** ("brain float") is another 16-bit format with the **same range as FP32** but fewer significant digits. Tiny gradients don't underflow, so no loss scaling is needed. It's the default on modern GPUs and TPUs.

## 3. Deep learning software

**Why use a framework** like PyTorch? Three reasons (from the lecture):
1. **Quick to try new ideas.**
2. **Gradients are computed for you** (automatic backprop, Lecture 6).
3. **Runs efficiently on GPUs.**

The best way to understand a framework is to build a tiny one. In Lecture 6 we built an autograd engine for single numbers. Now let's do it for **arrays**, with a PyTorch-like interface.

### 3.1 A Tensor that knows how to backprop
Each operation computes its result with NumPy, and also saves a small function that applies its backward rule (upstream × local, as in Lecture 6).

In [ ]:
class Tensor:
    def __init__(self, data, parents=()):
        self.data = np.array(data, dtype=float)
        self.grad = np.zeros_like(self.data)
        self.parents = parents
        self.backward_rule = lambda: None

    def __matmul__(self, other):                       # matrix multiply: self @ other
        out = Tensor(self.data @ other.data, parents=(self, other))
        def backward_rule():
            self.grad += out.grad @ other.data.T       # dL/dx = dL/dy W^T
            other.grad += self.data.T @ out.grad       # dL/dW = x^T dL/dy
        out.backward_rule = backward_rule
        return out

    def __add__(self, bias):                           # add a bias vector to every row
        out = Tensor(self.data + bias.data, parents=(self, bias))
        def backward_rule():
            self.grad += out.grad
            bias.grad += out.grad.sum(axis=0)          # the bias was used by every row, so its gradients add up
        out.backward_rule = backward_rule
        return out

    def relu(self):
        out = Tensor(np.maximum(0, self.data), parents=(self,))
        def backward_rule():
            self.grad += out.grad * (self.data > 0)
        out.backward_rule = backward_rule
        return out

The loss and `backward()` (the same topological-order sweep as Lecture 6):

In [ ]:
def cross_entropy(scores, labels):
    # Softmax cross-entropy loss, as a Tensor with its own backward rule.
    shifted = scores.data - scores.data.max(axis=1, keepdims=True)
    probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
    N = len(labels)
    out = Tensor(-np.mean(np.log(probs[np.arange(N), labels])), parents=(scores,))
    def backward_rule():
        d = probs.copy()
        d[np.arange(N), labels] -= 1                   # probabilities minus one-hot labels
        scores.grad += d / N
    out.backward_rule = backward_rule
    return out

def backward(loss):
    order, visited = [], set()
    def visit(t):
        if id(t) not in visited:
            visited.add(id(t))
            for parent in t.parents:
                visit(parent)
            order.append(t)
    visit(loss)
    loss.grad = np.ones_like(loss.data)
    for t in reversed(order):
        t.backward_rule()

### 3.2 Layers, an optimizer, and a data loader

In [ ]:
class Linear:
    def __init__(self, n_in, n_out, rng):
        self.W = Tensor(rng.normal(0, np.sqrt(2 / n_in), size=(n_in, n_out)))
        self.b = Tensor(np.zeros(n_out))
    def __call__(self, x):
        return x @ self.W + self.b
    def parameters(self):
        return [self.W, self.b]

class SGD:
    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr
    def zero_grad(self):                               # clear the old gradients
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)
    def step(self):                                    # take one gradient-descent step
        for p in self.parameters:
            p.data -= self.lr * p.grad

def data_loader(X, y, batch_size, rng):
    # Give back shuffled minibatches, covering the data once.
    order = rng.permutation(len(X))
    for start in range(0, len(X), batch_size):
        batch = order[start:start + batch_size]
        yield X[batch], y[batch]

### 3.3 Train a network with it
The training loop has **exactly** the shape of a PyTorch training loop: forward, loss, `zero_grad`, `backward`, `step`.

In [ ]:
digits = load_digits()
X_all = digits.data / 16.0 - 0.3
y_all = digits.target
order = np.random.default_rng(0).permutation(len(X_all))
X_train, y_train = X_all[order[:1300]], y_all[order[:1300]]
X_test, y_test = X_all[order[1300:]], y_all[order[1300:]]

rng = np.random.default_rng(1)
layer1 = Linear(64, 100, rng)
layer2 = Linear(100, 10, rng)
optimizer = SGD(layer1.parameters() + layer2.parameters(), lr=0.1)

for epoch in range(10):
    for X_batch, y_batch in data_loader(X_train, y_train, 64, rng):
        scores = layer2(layer1(Tensor(X_batch)).relu())    # forward
        loss = cross_entropy(scores, y_batch)
        optimizer.zero_grad()                              # clear old gradients
        backward(loss)                                     # backprop
        optimizer.step()                                   # update

test_scores = layer2(layer1(Tensor(X_test)).relu())
print(f"test accuracy: {np.mean(np.argmax(test_scores.data, axis=1) == y_test):.1%}")

About a hundred lines, and we have a working mini-framework. The real PyTorch adds thousands of operations and GPU support, but `.backward()` is the same idea.

**Why `zero_grad()`?** Gradients are **added** (`+=`) into `.grad`, so if you don't clear them, each step's gradient piles on top of the last. Forgetting this is a classic PyTorch bug:

In [ ]:
w = Tensor([[1.0]])
for step in range(3):
    loss = Tensor([[2.0]]) @ w           # d(loss)/dw = 2 every time
    backward(loss)                       # ...but we never clear w.grad
    print(f"step {step + 1}: w.grad = {w.grad.item()}  (should be 2)")

## 4. Static vs. dynamic graphs

Our mini-framework builds the graph **as the code runs**, fresh every time. That's a **dynamic** graph (PyTorch's approach): ordinary Python `if`s and loops just work, and debugging is easy.

**Static** graphs (TensorFlow 1.x) build the whole graph **first**, then run it many times. Advantages: the framework can **optimize** the whole graph before running it (e.g., fuse several operations into one), and **save** it to run without Python (on servers or phones).

Modern PyTorch can do both: write normal code, then **compile** or **trace** it (`torch.compile`, `torch.jit.trace`).

**A pitfall of tracing:** a tracer runs your function once on an example input and **records the operations that happened**. If the code has an `if` that depends on the data, only the branch the example took gets recorded. Here's a tiny tracer:

In [ ]:
def trace(function, example_input):
    # Run `function` once and record which operations it performed.
    recording = []
    class Recorder:
        def __init__(self, value): self.value = value
        def __mul__(self, c): recording.append(("multiply", c)); return Recorder(self.value * c)
        def __add__(self, c): recording.append(("add", c)); return Recorder(self.value + c)
        def __neg__(self): recording.append(("negate", None)); return Recorder(-self.value)
        def sum(self): return self.value.sum()               # an ordinary number: the tracer can't see the `if`
    function(Recorder(example_input))
    return recording

def replay(recording, x):
    # Run the recorded operations on a new input, without the original Python code.
    for op, c in recording:
        x = x * c if op == "multiply" else x + c if op == "add" else -x
    return x

def f(x):
    if x.sum() > 0:
        return x * 2 + 1
    return -x

recorded = trace(f, np.array([1.0, 2.0]))          # traced with a POSITIVE example
print("recorded operations:", recorded)
print("negative input, real function: ", -np.array([-3.0, -4.0]))
print("negative input, traced graph:  ", replay(recorded, np.array([-3.0, -4.0])), " <- wrong!")

The traced graph only knows the positive branch, so it silently gives the wrong answer for negative inputs.

| | Static graph | Dynamic graph |
|---|---|---|
| Examples | TensorFlow 1.x, compiled PyTorch | PyTorch, TensorFlow 2 (eager mode) |
| `if` statements and loops | special graph operations | plain Python |
| Debugging | harder | easy |
| Optimization, deployment | easier | needs a compile/export step |

## 5. PyTorch, TensorFlow, JAX

* **PyTorch:** dynamic graphs, Pythonic; dominant in research. Levels of abstraction: **tensors** (NumPy-like, can run on GPU), **autograd** (`.backward()`), **`nn.Module`** (layers), **`torch.optim`** (optimizers), **`DataLoader`**, pretrained models (`torchvision.models`).
* **TensorFlow:** version 1 used static graphs; **version 2** is dynamic by default, with **Keras** as its high-level API.
* **JAX:** NumPy-like functions plus `grad` (differentiate), `jit` (compile), `vmap` (vectorize); popular for large-scale training on TPUs.

Here's our mini-framework's training loop in real PyTorch. Notice it's almost identical:

In [ ]:
# [PyTorch] Needs: pip install torch
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Sequential(nn.Linear(64, 100), nn.ReLU(), nn.Linear(100, 10)).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train)), batch_size=64, shuffle=True)

for epoch in range(10):
    for X_batch, y_batch in loader:
        scores = model(X_batch.to(device))
        loss = nn.functional.cross_entropy(scores, y_batch.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
print("final loss:", loss.item())

## 6. Training on many GPUs

When one GPU is too slow, or the model doesn't fit:
* **Data parallelism:** every GPU has a **copy of the model** and processes a **different part of the batch**. The gradients are **averaged** across GPUs, so all copies stay identical.
* **Model parallelism:** the model itself is split across GPUs (by layers, or by splitting big matrices).

**Data parallelism gives exactly the same gradient as one big batch.** The batch gradient is an average over examples, so averaging the gradients of 4 equal parts is the same thing. Check, with 4 pretend GPUs:

In [ ]:
def gradient(W, X, y):
    # Gradient of the softmax loss for a linear classifier (Lecture 3).
    scores = X @ W
    probs = np.exp(scores - scores.max(1, keepdims=True))
    probs /= probs.sum(1, keepdims=True)
    probs[np.arange(len(y)), y] -= 1
    return X.T @ probs / len(y)

W = np.random.default_rng(2).normal(size=(64, 10))
X_batch, y_batch = X_train[:256], y_train[:256]

one_big_batch = gradient(W, X_batch, y_batch)
four_gpus = [gradient(W, X_batch[k * 64:(k + 1) * 64], y_batch[k * 64:(k + 1) * 64]) for k in range(4)]
print("average of 4 GPUs' gradients == big-batch gradient:", np.allclose(np.mean(four_gpus, axis=0), one_big_batch))

**How much memory does training need?** Much more than the weights alone. With the Adam optimizer in mixed precision, each parameter needs about **16 bytes**: 16-bit weights and gradients (2 + 2), plus 32-bit copies of the weights and Adam's two running averages (4 + 4 + 4).

In [ ]:
for name, n_params in [("ResNet-50", 25.6e6), ("a 7-billion-parameter model", 7e9)]:
    print(f"{name:28s}: about {16 * n_params / 1e9:.1f} GB just for the training state")
print("(one H100 GPU has 80 GB, so big models must be split across GPUs)")

## Summary
* **GPUs** have thousands of simple cores; deep learning is mostly parallel matrix math, so it runs much faster on them.
* Matrix multiplies are **compute-bound** (fast); elementwise ops are **memory-bound** (slow).
* **16-bit** numbers halve memory and unlock tensor cores; FP16 needs **loss scaling**, BF16 doesn't.
* A framework = **tensors + autograd + layers + optimizers + data loading + GPU support**. Ours fit in about a hundred lines.
* **Dynamic** graphs are easy to write and debug; **static** graphs are easier to optimize and deploy. Tracing freezes `if` statements.
* **Data parallelism** splits the batch across GPUs and averages the gradients: the same result as one big batch.

### Check your understanding
1. Why are GPUs good for deep learning?
2. Why does FP16 training need loss scaling?
3. What goes wrong if you forget `optimizer.zero_grad()`?
4. Why does tracing a function with an `if` statement sometimes give wrong answers?

<details><summary>Answers</summary>

1. Deep learning is mostly matrix multiplication, which splits into many independent dot products that thousands of GPU cores can compute at once.
2. FP16 can't represent very small numbers, so tiny gradients become 0; scaling the loss scales them up into range.
3. Gradients add up across steps, so each update uses the sum of all previous gradients.
4. The tracer only records the branch the example input took.
</details>

**Next:** Lecture 10, training neural networks (part 1).